# roadstyle — example maps

This notebook builds the example maps from a small bundled dataset of real OSM road edges (Sundbyberg, Sweden). Run it top-to-bottom (Kernel → Restart & Run All) to regenerate the maps in `notebooks/output/`.

**What it shows**
1. The canonical input (`RoadEdges` / `normalize_edges`)
2. Classic OSM styling (palettes & themes) — **folium** backend
3. The **lonboard** (WebGL) backend — a different output type
4. Filtering by road type
5. Palettes as editable JSON
6. The styler engine (class / categorical / numeric)

> Maps are written to disk (`.save()` for folium, `.to_html()` for lonboard) to keep this notebook small.

## 0. Setup
Imports and an `output/` folder for the rendered maps.

In [ ]:
from pathlib import Path
import geopandas as gpd
import roadstyle as rs

print('roadstyle', rs.__version__)

DATA = Path('data/sundbyberg_edges.gpkg')
OUT = Path('output'); OUT.mkdir(exist_ok=True)
print('data exists:', DATA.exists())

## 1. The canonical input — `RoadEdges`

roadstyle's input contract is a set of road edges normalised to **EPSG:4326** with **line geometry** and a **class column** (default `highway`). You load your data however you like (here a GeoPackage); `normalize_edges()` converts any GeoDataFrame into that canonical shape (reprojecting, dropping non-line geometry, optionally renaming your class column). `render_edges()` also accepts a plain GeoDataFrame and coerces it for you.

In [ ]:
raw = gpd.read_file(DATA)
print('raw  :', len(raw), 'edges | CRS', raw.crs.to_epsg(), '| columns', list(raw.columns))

edges = rs.normalize_edges(raw, class_col='highway')
print('canon:', len(edges), 'edges | CRS', edges.gdf.crs.to_epsg(),
      '| class_col', repr(edges.class_col))
print('road types:', sorted(edges.gdf['highway'].dropna().unique().tolist()))

## 2. Classic OSM styling — folium backend

`render_edges()` returns a **`folium.Map`** (the default backend). The **highsat** palette is the high-contrast theme (cyan motorway, pink trunk, …); **carto** mimics the classic OSM look. Themes (`light`/`dark`/`satellite`) swap the road casing and default base map. Use `.save()` to write a standalone interactive HTML file.

In [ ]:
m1 = rs.render_edges(edges, palette='highsat', theme='dark')
print('folium output ->', type(m1).__module__ + '.' + type(m1).__name__)
m1.save(str(OUT / 'highsat_dark.html'))

m2 = rs.render_edges(edges, palette='carto', theme='light')
m2.save(str(OUT / 'carto_light.html'))

print('saved:', sorted(p.name for p in OUT.glob('*.html')))
# In a live notebook you can also display it inline simply by ending a cell with: m1

## 3. The lonboard (WebGL) backend — a different output type

Pass `backend='lonboard'` to get a **`lonboard.Map`** instead of a `folium.Map`. lonboard renders with the GPU (deck.gl), so it stays smooth for very large edge sets where folium would struggle. It is a Jupyter widget — display it by ending a cell with the map object — and you can write it to a standalone file with `.to_html()`.

The two backends share the **same styling engine** (the geometry sandwich = two layers: casing under, fill over), so the cartography matches; only the rendering technology differs.

In [ ]:
try:
    gpu = rs.render_edges(edges, backend='lonboard', theme='dark')
    print('lonboard output ->', type(gpu).__module__ + '.' + type(gpu).__name__)
    print('layers (casing + fill):', len(gpu.layers))
    gpu.to_html(str(OUT / 'lonboard_dark.html'))
    print('saved output/lonboard_dark.html')
    # display inline in a live notebook by ending the cell with:  gpu
except ImportError:
    print('lonboard not installed — `pip install lonboard` (or the [lonboard] extra)')

## 4. Filter by road type

`include`/`exclude` keep or drop road classes before rendering. `match_links=True` means `primary` also matches `primary_link`.

In [ ]:
major = rs.render_edges(edges, theme='satellite', basemap='satellite',
                        include=['motorway', 'trunk', 'primary', 'secondary'])
major.save(str(OUT / 'major_satellite.html'))
print('major-roads map saved')

## 5. Palettes as editable JSON

Built-in palettes are typed Python (the tested defaults), but any palette can be exported to **JSON**, hand-edited, and reloaded — one colour source that Python *and* a web frontend can share. `load_palette()` auto-registers by name so `render_edges(palette=name)` can use it.

In [ ]:
# export -> edit -> reload
rs.save_palette('highsat', str(OUT / 'highsat.json'))

import json
doc = json.load(open(OUT / 'highsat.json'))
doc['name'] = 'my_custom'
doc['roads']['motorway']['fill'] = '#FF0000'   # motorway -> red
json.dump(doc, open(OUT / 'my_custom.json', 'w'), indent=2)

rs.load_palette(OUT / 'my_custom.json')          # registers 'my_custom'
custom = rs.render_edges(edges, palette='my_custom', theme='dark')
custom.save(str(OUT / 'custom_palette.html'))
print('custom palette registered:', 'my_custom' in rs.PALETTES)

## 6. The styler engine

Under the hood every styling mode resolves a whole GeoDataFrame to per-edge colours/widths via a **Styler**. `ClassStyler` is the OSM class styler; `CategoricalStyler` colours by a discrete column (e.g. congestion levels); `NumericStyler` colours by a numeric column (e.g. traffic volume) with a continuous ramp. Here we inspect the resolved colours directly.

In [ ]:
from collections import Counter
rf = rs.ClassStyler().resolve_frame(edges.gdf, theme='dark')
print('class styler — colour counts (top 6):')
for hexc, n in Counter(rf.fill).most_common(6):
    print(f'  {hexc}  {n:5d} edges')

In [ ]:
# Categorical + numeric on a tiny synthetic sample (no extra data needed)
from shapely.geometry import LineString
sample = gpd.GeoDataFrame(
    {'congestion': ['low', 'moderate', 'heavy', 'severe', None],
     'aadt':       [200,    4000,       12000,   25000,    None]},
    geometry=[LineString([(i, 0), (i + 1, 1)]) for i in range(5)], crs=4326)

cat = rs.CategoricalStyler(column='congestion',
        colors={'low': '#11D68F', 'moderate': '#FFCF43',
                'heavy': '#F24E42', 'severe': '#A92727'}).resolve_frame(sample)
print('categorical fills:', cat.fill)

num = rs.NumericStyler(column='aadt', cmap='viridis',
        vmin=0, vmax=25000, width_by=(1, 8)).resolve_frame(sample)
print('numeric fills    :', num.fill)
print('numeric widths   :', [round(w, 2) for w in num.width])
print('legend           :', num.legend['kind'], num.legend['title'],
      num.legend['vmin'], '->', num.legend['vmax'])

## Done

All maps are in `notebooks/output/`. Open any `.html` in a browser for the interactive map. folium maps have a hover-highlight, a road-type filter panel and a base-layer switcher; the lonboard map is GPU-rendered for large data.

**Output types seen here:** `folium.Map` (sections 2, 4, 5) and `lonboard.Map` (section 3). A stack-agnostic JSON spec for embedding in a custom website is coming in a later phase.

_Data-driven maps (`render_edges(color_by=...)`) render directly once the backend wiring lands; the engine in section 6 already computes the colours._